In [2]:
%%capture
!pip install llama-index llama-index-embeddings-openai qdrant-client llama-index-vector-stores-qdrant llama-index llama-index-llms-openai

In [5]:
import os
import sys
from getpass import getpass
import nest_asyncio

from IPython.display import Markdown, display

from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv("")

sys.path.append('../helpers')


ImportError: cannot import name 'IDF_EMBEDDING_MODELS' from 'qdrant_client.qdrant_fastembed' (C:\Users\anteb\anaconda3\Lib\site-packages\qdrant_client\qdrant_fastembed.py)

In [3]:
OPENAI_API_KEY = os.environ['OPENAI_API_KEY'] or getpass("Enter your OpenAI API key: ")
CO_API_KEY = os.environ.get('CO_API_KEY') or getpass("Enter CO_API_KEY: ")
QDRANT_API_KEY = os.environ['QDRANT_API_KEY'] or  getpass("Enter your Qdrant API Key:")

NameError: name 'os' is not defined

In [15]:
# QDRANT_URL = os.environ['QDRANT_URL'] or getpass("Enter your Qdrant URL:")

QDRANT_URL=":memory:"

In [17]:
from llama_index.core.settings import Settings
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.openai import OpenAIEmbedding

# Settings.llm = Cohere(model="command-r-plus", api_key=CO_API_KEY)

Settings.embed_model = OpenAIEmbedding(model_name="text-embedding-3-small")

In [18]:
from helpers.utils import setup_llm, setup_embed_model
from llama_index.core.settings import Settings


setup_llm(
    provider="openai",
    api_key=OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0.75,
    system_prompt="""Use ONLY the provided context and generate a complete, coherent answer to the user's query.
    Your response must be grounded in the provided context and relevant to the essence of the user's query.
    """
    )

setup_embed_model(
    provider="openai",
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
    )

ImportError: cannot import name 'IDF_EMBEDDING_MODELS' from 'qdrant_client.qdrant_fastembed' (C:\Users\anteb\anaconda3\Lib\site-packages\qdrant_client\qdrant_fastembed.py)

In [ ]:
import random
from llama_index.core.storage.docstore import SimpleDocumentStore

documents = get_documents_from_docstore("../naive_rag/data/raw")

random.seed(42)

documents_by_author = group_documents_by_author(documents)

senpai_documents = sample_documents(documents_by_author, num_samples=10)

In [ ]:
from naive_rag.helpers.IngestionCacheManager import SmartIngestionCache
from qdrant_client import QdrantClient
from llama_index.core.ingestion import IngestionCache, IngestionPipeline
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.qdrant import QdrantVectorStore


client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="csv_articles",)

ingest_cache = SmartIngestionCache().get_cache()

# create pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=256, chunk_overlap=16),
        Settings.embed_model
    ],
    docstore=SimpleDocumentStore(),
    vector_store=vector_store,
    cache=ingest_cache,
)

# run the pipeline
nodes = pipeline.run(documents = documents)